In [8]:
# load “Imports + config” notebook
%run ./0_setup.ipynb

In [9]:
df = pd.read_csv(VILL_IN_PATH)

vill_utm = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["xcoor"], df["ycoor"]),
    crs="EPSG:32648"
)

water = gpd.read_file(WATER_PATH)


# Reproject waterways to match village CRS (meters)
water_m = water.to_crs(vill_utm.crs)

# Merge all waterways into one geometry (fast distance computation)
water_union = water_m.union_all()

# Distance to nearest waterway (meters + km)
vill_utm["dist_to_water_m"] = vill_utm.geometry.distance(water_union)
vill_utm["dist_to_water_km"] = vill_utm["dist_to_water_m"] / 1000.0

# Save new distances

vill_utm.drop(columns="geometry").to_csv(VILL_OUT_PATH, index=False)

print("Saved:", VILL_OUT_PATH)
print(vill_utm[["vill_code", "dist_to_water_m", "dist_to_water_km"]].head())

Saved: /Users/sam/Desktop/CERDIProject/TermiteMound/Script/RawData/TM_Village_with_geo_climate_1km_waterdist.csv
   vill_code  dist_to_water_m  dist_to_water_km
0    4080109      3588.387109          3.588387
1    4080107       836.402968          0.836403
2    4080301       516.699846          0.516700
3    4080302       990.912171          0.990912
4    4080306       278.989734          0.278990
